# Week 7 · Monday: Multi-layer Perceptrons, Backprop and a Digit Classifier

On Friday one sigmoid neuron learned its own weights with `fit`, and we used `loss="binary_crossentropy"` as a setting without opening it. Today we open it, see why one neuron is not enough, watch a network of neurons learn, and build a whole model project from start to finish:

1. **Loss:** one number for how wrong a model is
2. **Gradient descent:** small steps downhill, and the learning rate that sets their size
3. **The limit of one perceptron:** XOR, four points no straight line separates
4. **Multi-layer perceptrons:** one hidden layer solves XOR
5. **Backpropagation:** how every weight gets its share of the loss
6. **Build: an MLP digit classifier:** problem, dataset, split, preprocessing, model, training
7. **Evaluating on the validation data:** confusion matrix, precision, recall, F1 and the wrong digits
8. **One improvement attempt and the test set:** one change measured, the test set once, limitations
9. **TensorBoard:** the training curves in the Scalars tab

Parts marked *Going deeper* are optional on a first pass: run their cells, then come back to them with the notes page.

Run the next cell first. It holds the imports and plotting helpers we reuse all day, and it fixes the random seeds so your numbers match the ones on the screen. MNIST downloads later, in section 6.2.

In [ ]:
# ---------- Setup and plotting helpers (run once; no need to read live) ----------
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import sklearn
import tensorflow as tf

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)   # seeds Python, NumPy and TensorFlow together


def plot_points(X, labels=None, title="", xlabel="Feature 1", ylabel="Feature 2",
                names=None, centers=None, ax=None):
    """Scatter 2D points, coloured by label if given (a label of -1 is drawn as a black x)."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    if labels is None:
        ax.scatter(X[:, 0], X[:, 1], s=45, edgecolor="black", alpha=0.85)
    else:
        for value in np.unique(labels):
            mask = labels == value
            if value == -1:
                ax.scatter(X[mask, 0], X[mask, 1], s=70, marker="x", color="black", label="Noise (-1)")
            else:
                label = names[value] if names is not None else f"{value}"
                ax.scatter(X[mask, 0], X[mask, 1], s=45, edgecolor="black", alpha=0.85, label=label)
        ax.legend()
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], s=300, marker="X", color="red",
                   edgecolor="black", linewidth=2, label="Centroids")
        ax.legend()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    if show:
        plt.show()


def plot_decision_boundary(model, X, y, title="", xlabel="Feature 1", ylabel="Feature 2",
                           names=None, ax=None):
    """Colour the background by what the model predicts at every point of a fine grid."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
    grid = np.c_[xx.ravel(), yy.ravel()].astype("float32")
    if isinstance(model, tf.keras.Model):                 # Keras returns probabilities: turn them into labels
        probabilities = model.predict(grid, verbose=0)
        if probabilities.shape[1] == 1:
            grid_predictions = (probabilities[:, 0] > 0.5).astype(int)    # the same cut as Keras's accuracy
        else:
            grid_predictions = probabilities.argmax(axis=1)
    else:
        grid_predictions = model.predict(grid)
    grid_predictions = grid_predictions.reshape(xx.shape)
    classes = np.unique(y)
    colours = ListedColormap(plt.cm.tab10.colors[:len(classes)])   # same colours as the points
    ax.contourf(xx, yy, grid_predictions, levels=np.arange(len(classes) + 1) - 0.5, alpha=0.25, cmap=colours)
    plot_points(X, y, title, xlabel, ylabel, names=names, ax=ax)
    if show:
        plt.show()


def plot_history(history, metric="loss", title=""):
    """Training and validation curves of one metric against the epoch, on one chart."""
    epochs = range(1, len(history.history[metric]) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history.history[metric], marker="o", markersize=3, linewidth=2, label="training")
    plt.plot(epochs, history.history["val_" + metric], marker="o", markersize=3, linewidth=2, label="validation")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


def show_image_grid(images, titles, columns=5):
    """Draw digits (784-long rows or 28 by 28 by 1 arrays) as 28 by 28 images, in rows of `columns`."""
    rows = int(np.ceil(len(images) / columns))
    plt.figure(figsize=(2.2 * columns, 2.4 * rows))
    for i, (image, title) in enumerate(zip(images, titles)):
        plt.subplot(rows, columns, i + 1)
        plt.imshow(np.reshape(image, (28, 28)), cmap="gray")
        plt.title(title, fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


print(f"Python {sys.version.split()[0]}, TensorFlow {tf.__version__}, Keras {tf.keras.__version__}, "
      f"scikit-learn {sklearn.__version__}")
print("Setup complete")

## 1. Loss: one number for how wrong

Every training row has a **target**: the right answer, taken from the data. What the model says for that row is its **prediction**. A **loss function** turns the gap between the predictions and the targets into one number, and training tries to make that number smaller. Lower is better, and 0 means every prediction was exactly right.

Which loss you use depends on the kind of answer the model gives: a number or a class.

### 1.1 Mean squared error, when the answer is a number

When the answer is a number, such as a price or a temperature, the usual loss is **mean squared error (MSE)**: take each miss (prediction minus target), square it, and average the squares.

Five made-up targets and five predictions. We compute the MSE with numpy, then with Keras's `tf.keras.losses.MeanSquaredError()`, to check they agree.

In [ ]:
y_true = np.array([3.0, 5.0, 2.5, 7.0, 4.0])
y_pred = np.array([2.5, 5.0, 4.0, 8.0, 3.0])      # Try changing it: make the 8.0 a 7.0

squared_misses = (y_pred - y_true) ** 2
print(f"Squared misses: {squared_misses}")
print(f"numpy MSE: {np.mean(squared_misses):.3f}")

mse = tf.keras.losses.MeanSquaredError()
print(f"Keras MSE: {float(mse(y_true, y_pred)):.3f}")

- The squared misses add up to 4.5, and 4.5 divided by 5 is 0.9. numpy and Keras agree: the Keras loss is this formula.
- Squaring does two jobs. A miss above and a miss below both count as positive, so they cannot cancel out. And a big miss costs far more than a small one: the third prediction is 1.5 off and costs 2.25, the first is 0.5 off and costs 0.25. Three times the miss, nine times the cost.
- A loss on its own says little. It is read against another loss on the same rows, which is how section 8 picks between two models.
- Try changing the 8.0 to 7.0, so the fourth prediction is exact. The squared misses add up to 3.5 and the MSE drops to 0.7. Put it back to 8.0.
- In Keras, a model that predicts a number compiles with `loss="mse"`.

**In an interview:** "A loss function turns a model's errors into one number that training makes smaller. For a numeric answer I use mean squared error, the average of the squared misses, so big misses cost the most."

### 1.2 Cross-entropy, when the answer is a class

When the answer is a class, the model outputs a **probability**: a number from 0 to 1 for how likely it thinks each class is. **Cross-entropy** asks how much probability the model gave to the true class. With two classes it is called **binary cross-entropy**, and for a row whose true label is 1 it is `-log(p)`, where `p` is the predicted probability of 1 and `log` is the natural logarithm (`np.log`).

One true label of 1, predicted three ways: 0.9 (confident and right), 0.6 (unsure) and 0.1 (confident and wrong). A Keras loss averages over all the rows it is given, so we call it once per prediction, `bce([[1.0]], [[p]])`, to get one number each.

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy()

for p in [0.9, 0.6, 0.1]:                     # Try changing it: [0.9, 0.6, 0.1, 0.99, 0.01]
    by_hand = -np.log(p)
    keras_value = float(bce([[1.0]], [[p]]))
    print(f"true 1, predicted {p}: numpy {by_hand:.3f}, Keras {keras_value:.3f}")

all_at_once = float(bce([[1.0], [1.0], [1.0]], [[0.9], [0.6], [0.1]]))
print(f"All three in one call: {all_at_once:.3f}")

- Confident and right costs 0.105, unsure costs 0.511, and confident and wrong costs 2.303, about twenty times the first. That steep cost for a confident mistake is why cross-entropy is the loss for classes.
- The last line passes all three rows in one call and gets their mean, 0.973. That is what a Keras loss does with a batch.
- Friday's `loss="binary_crossentropy"` was this number, averaged over each batch of rows.
- `metrics=["accuracy"]` is reported each epoch for us to read. The number training lowers is the loss.
- Try changing the list to add 0.99 and 0.01. A confident right answer at 0.99 costs 0.010, and a confident wrong one at 0.01 costs 4.605. Put the list back.

**In an interview:** "For a class I use cross-entropy. It is small when the model gives the true class a high probability and grows fast when the model is confident and wrong; Keras calls the two-class version `binary_crossentropy`."

## 2. Gradient descent

The loss says how wrong the model is. **Gradient descent** says how to change the weights to make it less wrong: repeatedly nudge each weight a small step in the direction that lowers the loss.

### 2.1 Downhill on one weight

The smallest model there is: one weight `w`, and the prediction is `w * x`. The data is four points with `y = 2.0 * x`, so the best weight is 2. For every candidate `w` we can compute the MSE of section 1.1. Drawn against `w`, those losses make a bowl.

In [ ]:
x = np.array([1.0, 2.0, 3.0, 4.0])
y = 2.0 * x                                   # the best weight is 2

def line_loss(w):
    """MSE of the model w * x against y."""
    return np.mean((w * x - y) ** 2)

candidates = np.linspace(-1, 5, 121)
plt.figure(figsize=(8, 5))
plt.plot(candidates, [line_loss(c) for c in candidates], linewidth=2)
plt.title("The loss of w * x for every w: a bowl")
plt.xlabel("w")
plt.ylabel("MSE")
plt.grid(alpha=0.3)
plt.show()

The bottom of the bowl is at `w = 2`, where the loss is 0. Training never sees the whole bowl. It only knows the loss where it stands, and the **slope** there: how much the loss changes when `w` moves a little. A weight's slope is also called its **gradient**, which is where gradient descent gets its name. A positive slope means the loss rises as `w` grows, so `w` should shrink. A negative slope means `w` should grow.

We find the slope with a **centred nudge**: move the weight a hair each way and see how much the loss changes.

```
slope = (loss(w + 0.001) - loss(w - 0.001)) / 0.002
```

Then one step of gradient descent is:

```
w = w - learning_rate * slope
```

The cell starts at `w = 0.0`, uses a learning rate of 0.05, takes five steps and marks each one on the bowl.

In [ ]:
def slope(w):
    """The centred nudge: how much the loss changes per unit of w, around w."""
    return (line_loss(w + 0.001) - line_loss(w - 0.001)) / 0.002

learning_rate = 0.05
w = 0.0                                       # Try changing it: 4.0
path = []
for step in range(6):
    print(f"step {step}: w = {w:.3f}, loss = {line_loss(w):.3f}, slope = {slope(w):.3f}")
    path.append(w)
    w = w - learning_rate * slope(w)          # one step against the slope

plt.figure(figsize=(8, 5))
plt.plot(candidates, [line_loss(c) for c in candidates], linewidth=2, alpha=0.5)
plt.plot(path, [line_loss(p) for p in path], "o-", color="red", label="the steps")
plt.title("Gradient descent on the bowl, learning rate 0.05")
plt.xlabel("w")
plt.ylabel("MSE")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

- At `w = 0` every prediction is 0, the loss is 30 and the slope is -30. Subtracting a negative slope adds to `w`: 0.05 times 30 is 1.5, so the first step lands on `w = 1.5`, where the loss is 1.875.
- Then 1.875, 1.969, 1.992 and 1.998. The steps shrink on their own: near the bottom the bowl flattens, the slope shrinks, and a step is the learning rate times the slope. `w` settles at 2 without ever being told where 2 is.
- Try changing the start to 4.0. The slope is now +30, the loss is 30 again (the bowl is symmetric), and `w` walks down to 2 from the other side: 4.0, 2.5, 2.125. Put it back to 0.0.
- Keras's `tf.keras.optimizers.SGD(learning_rate=...)` is this same plain update, applied after each batch. Friday's `"adam"` is a refined version of it, met in section 5.1.

**In an interview:** "Gradient descent lowers the loss step by step. At each step it takes the slope of the loss for each weight and moves the weight a little against it: the weight minus the learning rate times the slope."

### 2.2 The learning rate is the step size

The **learning rate** multiplies the slope before the step is taken, so it sets the size of every step. Same data, same start at `w = 0.0`, ten steps at three learning rates. The loss axis is on a log scale, where each equal step up the axis multiplies the loss by the same factor, because the three runs end up millions of times apart.

In [ ]:
rates = [0.01, 0.05, 0.2]                     # Try changing it: [0.1, 0.15]

plt.figure(figsize=(8, 5))
for rate in rates:
    w = 0.0
    losses = [line_loss(w)]
    for step in range(10):
        w = w - rate * slope(w)
        losses.append(line_loss(w))
    print(f"rate {rate}: w = {w:.3f}, loss = {line_loss(w):.1f}")
    plt.plot(range(11), losses, marker="o", linewidth=2, label=f"learning rate {rate}")
plt.yscale("log")
plt.title("Loss after each step, three learning rates")
plt.xlabel("Step")
plt.ylabel("MSE (log scale)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

- **Too small (0.01):** training crawls. After ten steps `w` is only at 1.606 and the loss is still 1.2.
- **About right (0.05):** `w` reaches 2.000 and the loss falls to 0.0.
- **Too large (0.2):** every step overshoots the bottom of the bowl and lands higher up the far side. The first step takes `w` from 0 to 6, four past the target, and every later step doubles the distance and flips the side. After ten steps `w` is -2046.000 and the loss is 31,457,280. The run has **diverged**: the loss grows instead of falling.
- Try changing the rates to 0.1 and 0.15. Where does it stop converging? At 0.1 the weight bounces from side to side but closes in on 2 (`w = 1.998`). At 0.15 it bounces further out every step (`w = -16.626`, loss 2602.1). The edge lies between them. Put the rates back.
- On a real model you read the loss curve the same way. Falling slowly and still falling at the end: try a larger rate or more epochs. Bouncing or climbing: try a smaller rate.

**In an interview:** "The learning rate is the step size of gradient descent. Too small and training crawls; too large and each step overshoots, so the loss bounces or grows. I choose it by reading the loss curve."

## 3. The limit of one perceptron: XOR

Friday's perceptron drew one straight line and labelled each point by the side it fell on. That makes it a **linear classifier**. Two classes are **linearly separable** when one straight line puts them on opposite sides, and Friday's blobs were.

**XOR** ("exclusive or") is the classic table that is not. Its label is 1 when exactly one of its two inputs is 1. Look at the four points before training anything.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype="float32")
y_xor = np.array([0, 1, 1, 0])                # 1 when exactly one input is 1

for row, label in zip(X_xor.astype(int), y_xor):
    print(f"x = {row}  label = {label}")

fig, ax = plt.subplots(figsize=(6, 5))
plot_points(X_xor, y_xor, "XOR: the 1s and the 0s sit on opposite diagonals", "x1", "x2", ax=ax)
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
plt.show()

The two 1s sit on one diagonal of the square and the two 0s on the other. Any straight line that puts both 1s on one side takes at least one 0 with them, so the best any line can do is three points out of four: accuracy 0.75.

Now Friday's model, one sigmoid neuron, trained on these four rows. We give Adam a larger learning rate than its default (0.05 instead of 0.001) so that 300 epochs are plenty for four rows. The fit takes under a minute.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)   # the same starting weights on every rerun

neuron = tf.keras.Sequential([
    tf.keras.Input(shape=(2,)),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
neuron.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.05),
               loss="binary_crossentropy", metrics=["accuracy"])
neuron.fit(X_xor, y_xor, epochs=300, verbose=0)         # Try changing it: epochs=900

loss, accuracy = neuron.evaluate(X_xor, y_xor, verbose=0)
print(f"One neuron on XOR: loss {loss:.3f}, accuracy {accuracy}")
print(f"Probability of 1 for each point: {neuron.predict(X_xor, verbose=0).ravel().round(3)}")
plot_decision_boundary(neuron, X_xor, y_xor, "One sigmoid neuron on XOR", "x1", "x2")

- Every probability is 0.5 and the loss is 0.693, which is `-log(0.5)`: the cross-entropy of answering 0.5 for every point. The neuron has given up and hedges. For one neuron on XOR, that is the lowest loss it can reach.
- The accuracy prints 0.5. With every probability within a hair of 0.5, which side of the cut a point lands on comes down to rounding, so your copy may print 0.25, 0.5 or 0.75. It never prints 1.0. The background colouring uses the same cut as the accuracy.
- Whatever it predicts, its boundary is a straight line. The sigmoid bends the output into a probability, but the points where that probability crosses 0.5 still lie on one straight line.
- Try changing `epochs` to 900. The loss stays at 0.693 and the accuracy never reaches 1.0: more training cannot give one straight line a shape it cannot draw. Accuracy stuck below 1.0 on these four points means the model cannot draw the shape, and more epochs will not change that. Put it back to 300.

**In an interview:** "A single perceptron is a linear classifier: it draws one straight line. XOR's two classes sit on opposite diagonals, so no line separates them and the best it can do is three of four. A hidden layer with a non-linear activation fixes that."

## 4. Multi-layer perceptrons

A **multi-layer perceptron (MLP)** is a network of Dense layers with at least one **hidden layer** between the input and the output. A hidden layer's outputs are neither the inputs you give the model nor the answer you read from it: they are numbers the network makes for its own use.

### 4.1 One hidden layer solves XOR

The same neuron as section 3, with one hidden layer of 4 units in front of it. The hidden layer uses **tanh**, Friday's activation that squashes a sum into the range -1 to 1. Everything else stays as in section 3: Adam at 0.05, 300 epochs, the same seed.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)

xor_mlp = tf.keras.Sequential([
    tf.keras.Input(shape=(2,)),
    tf.keras.layers.Dense(4, activation="tanh"),          # the hidden layer. Try changing it: Dense(2, ...)
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
xor_mlp.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.05),
                loss="binary_crossentropy", metrics=["accuracy"])
history_xor = xor_mlp.fit(X_xor, y_xor, epochs=300, verbose=0)

loss, accuracy = xor_mlp.evaluate(X_xor, y_xor, verbose=0)
first_perfect = next((i + 1 for i, a in enumerate(history_xor.history["accuracy"]) if a == 1.0), "never")
print(f"Numbers this model learns: {xor_mlp.count_params()}")
print(f"MLP on XOR: loss {loss:.3f}, accuracy {accuracy}")
print(f"First epoch with all four points right: {first_perfect}")
print(f"Probability of 1 for each point: {xor_mlp.predict(X_xor, verbose=0).ravel().round(3)}")
plot_decision_boundary(xor_mlp, X_xor, y_xor, "One hidden layer of 4 tanh units on XOR", "x1", "x2")

- The model learns 17 numbers. A Dense layer holds one weight per input per unit plus one bias per unit: the hidden layer has 2 × 4 + 4 = 12 and the output layer 4 × 1 + 1 = 5.
- Accuracy 1.0: all four points right, where the single neuron stopped at 0.5. In this run it first got all four right in epoch 16, and by epoch 300 its probabilities sit near 0 for (0, 0) and (1, 1) and near 1 for the other two.
- The boundary is no longer one straight line: it bends, so the two diagonals land in different regions.
- **Why the hidden activation must be non-linear.** Friday showed that two layers with no activation collapse into one weighted sum. Delete `activation="tanh"` and this network draws one straight line again and fails XOR like the single neuron. The bend in tanh or ReLU is what the hidden layer adds.
- **Which activation goes where.** Hidden layers: `relu` or `tanh`. Output for two classes: `sigmoid`, one probability. Output for many classes: `softmax`, one probability per class.
- More hidden units fit more shapes, but on a small training set they also memorise its rows. Section 6.6 shows what that looks like.
- Try changing the hidden layer to 2 units. The count drops to 9 (2 × 2 + 2 = 6, plus 2 + 1 = 3). In this run, with this seed, the 2-unit network gets stuck at accuracy 0.5: fewer units give training fewer ways to find the shape. Put it back to 4 and rerun the cell, because section 4.2 reads this model.

**In an interview:** "A multi-layer perceptron is a stack of Dense layers with at least one hidden layer. The hidden layer's non-linear activation lets the network draw a bent boundary, which is why it solves XOR where one neuron cannot."

### 4.2 What the hidden layer learned

*Going deeper.* The idea to keep: the hidden layer re-describes the four points so that one straight cut separates them.

A layer of a Keras model can be called on its own, like a function. `xor_mlp.layers[0]` is the hidden layer: for each point it returns four numbers, one per tanh unit. The output neuron then takes a weighted sum `z` of those four numbers plus its bias, and its probability is above 0.5 exactly where `z` is above 0.

In [ ]:
hidden_layer, output_layer = xor_mlp.layers
hidden_values = hidden_layer(X_xor).numpy()               # four new numbers per point
kernel, bias = output_layer.get_weights()
z = hidden_values @ kernel[:, 0] + bias[0]                # the output neuron's weighted sum

for point, h, z_value, label in zip(X_xor.astype(int), hidden_values, z, y_xor):
    print(f"x = {point}  hidden = {h.round(2)}  z = {z_value:+.2f}  label = {label}")

- Each point goes in as two numbers and comes out of the hidden layer as four, each between -1 and 1.
- In those four numbers, `z` is positive for both points labelled 1 and negative for both points labelled 0. The output neuron is still one straight cut, made in the hidden layer's description of the points instead of the original one.
- A hidden layer is a change of description that training chooses, so that the last layer's job becomes easy. The digit network in section 6 does the same with 128 numbers per image.

## 5. Backpropagation

Gradient descent needs the slope of every weight. Section 2 had one weight. `xor_mlp` has 17, and the digit network in section 6 has 101,770. **Backpropagation** ("backprop") is the method that works out every weight's slope by passing the loss backward through the network, layer by layer.

### 5.1 The algorithm

One training step has four parts, and `fit` repeats them for every batch:

1. **Forward pass:** the batch goes through the layers, from input to output, and becomes predictions.
2. **Loss:** the predictions are compared with the targets, giving one number.
3. **Backward pass:** starting at the loss, each layer works out its own weights' share of the loss (their slopes) and hands the rest back to the layer before it.
4. **Update:** the optimizer moves every weight a small step against its slope.

```
forward pass -> loss -> backward pass (slopes) -> update -> repeat
```

- **Why backward.** The output layer's weights touch the loss directly. A hidden weight affects the loss only through the layers after it, so its slope is built from theirs, and those have to be worked out first.
- **Why not nudge every weight.** Section 2's centred nudge gives the same slopes, but it needs two extra forward passes per weight: 203,540 forward passes for one update of the digit network. Backprop gets every slope from one forward pass and one backward pass.
- **What it needs.** Every layer and the loss must have a slope. Friday's step function is flat everywhere except at its jump, so nudging a weight changes nothing and the slope comes out as 0, with no direction to move. That is why trained networks use sigmoid, tanh and ReLU.
- **Backprop finds the slopes; the optimizer takes the step.** Plain gradient descent (`SGD` in Keras) moves each weight by the learning rate times its slope. **Adam**, behind every `optimizer="adam"` this week, is gradient descent whose step size adapts for each weight. The learning rate lives in the optimizer.
- **In code.** `loss=` and `optimizer=` in `compile` name parts 2 and 4. Backprop has no line of its own: it runs inside `fit`, on every batch.

**In an interview:** "Backpropagation computes the slope of the loss for every weight by passing the error backward from the output, layer by layer. The optimizer then moves each weight against its slope, and `fit` repeats forward pass, loss, backward pass and update for every batch."

### 5.2 One step by hand on a tiny network

The smallest network that still has a hidden layer: one input, one hidden unit and one output, joined by two weights, with no bias and no activation, so only the bookkeeping is left. The row is input `x = 2.0` with target `1.0`; the weights are `w1 = 0.5` (input to hidden) and `w2 = 1.5` (hidden to output).

```
h          = w1 * x                    = 0.5 * 2.0 = 1.0
prediction = w2 * h                    = 1.5 * 1.0 = 1.5
loss       = (prediction - target) ** 2 = 0.5 ** 2  = 0.25
```

The prediction is too high, so both weights must shrink. Their slopes say by how much, and backprop builds them from the output backward:

- At the output, the loss changes by 1.0 for each unit the prediction moves.
- `w2` multiplies `h = 1.0` on its way to the prediction, so its slope is 1.0 × 1.0 = 1.0.
- A change in `w1` is multiplied by `x = 2.0` into `h`, then by `w2 = 1.5` out of it, so its slope is 1.0 × 1.5 × 2.0 = 3.0.

The cell measures both slopes with section 2's centred nudge, holding the other weight still, then takes one step at learning rate 0.02.

In [ ]:
x_row, target = 2.0, 1.0
w1, w2 = 0.5, 1.5

def pair_loss(w1, w2):
    """Squared error of the two-weight network on the one row."""
    h = w1 * x_row
    prediction = w2 * h
    return (prediction - target) ** 2

print(f"prediction {w2 * w1 * x_row}, loss {pair_loss(w1, w2)}")
slope_w2 = (pair_loss(w1, w2 + 0.001) - pair_loss(w1, w2 - 0.001)) / 0.002    # nudge w2, hold w1
slope_w1 = (pair_loss(w1 + 0.001, w2) - pair_loss(w1 - 0.001, w2)) / 0.002    # nudge w1, hold w2
print(f"slope w2 {slope_w2:.3f}, slope w1 {slope_w1:.3f}")

learning_rate = 0.02                          # Try changing it: 0.1
w2 = w2 - learning_rate * slope_w2
w1 = w1 - learning_rate * slope_w1
print(f"w2 {w2:.2f}, w1 {w1:.2f}, prediction {w2 * w1 * x_row:.2f}, loss {pair_loss(w1, w2):.3f}")

- The forward pass gives prediction 1.5 and loss 0.25, and the nudges print slopes of 1.000 for `w2` and 3.000 for `w1`: the numbers worked out above, measured instead of derived.
- `w1`'s slope is three times larger because its effect passes through `w2 = 1.5` and `x = 2.0` on the way to the prediction. That is the chain of slopes backprop follows: an earlier weight's slope is multiplied by what it passes through on the way back.
- One step: `w2 = 1.5 - 0.02 × 1.0 = 1.48` and `w1 = 0.5 - 0.02 × 3.0 = 0.44`. The forward pass again gives `h = 0.88`, prediction about 1.30 and loss about 0.091. One step lowered the loss from 0.25. `fit` does exactly this, for every weight and every batch.
- Try changing the learning rate to 0.1. Then `w2` is 1.40, `w1` is 0.20, the prediction 0.56 and the loss 0.194: the step jumped past the target to the other side, as the rate of 0.2 did in section 2.2. Put it back to 0.02.

**In an interview:** "On a two-weight network the forward pass gives prediction 1.5 and loss 0.25. Passing the loss back gives a slope of 1 for the output weight and 3 for the hidden weight, because its effect is multiplied by the output weight and the input on the way. One step at learning rate 0.02 lowers the loss to about 0.09."

### 5.3 TensorFlow finds the same slopes

*Going deeper.* The idea to keep: TensorFlow's backprop returns the same slopes as the nudge, 1.0 and 3.0, from one backward pass.

`tf.GradientTape` records the operations run inside its `with` block. `tape.gradient(loss, variables)` then runs backward through that record and returns one slope per variable, in the order the variables are listed. It is the machinery `fit` uses on every batch.

In [ ]:
w1_t = tf.Variable(0.5)
w2_t = tf.Variable(1.5)

with tf.GradientTape() as tape:                           # records the forward pass
    loss_t = (w2_t * (w1_t * 2.0) - 1.0) ** 2

grad_w2, grad_w1 = tape.gradient(loss_t, [w2_t, w1_t])    # one backward pass, both slopes
print(f"loss {float(loss_t):.2f}")
print(f"slope w2 {float(grad_w2):.1f}, slope w1 {float(grad_w1):.1f}")

- The same loss, 0.25, and the same slopes, 1.0 for `w2` and 3.0 for `w1`, with no nudging.
- Thursday's page writes a whole training loop around the tape for the digit network. Today `fit` runs it for us.

## 6. Build: an MLP digit classifier

Sections 6 to 8 are the shape of a whole model project, in the order your own project notebook follows: problem statement, dataset, split, preprocessing, model with a reason for each choice, training with both curves, evaluation, error analysis, one improvement attempt, the test set used once, and limitations.

### 6.1 Problem statement

**What is predicted:** the digit, 0 to 9, written in a small scanned image of one handwritten digit.

**Why it matters:** organisations that receive handwritten forms, such as a post office reading the numbers on envelopes or a bank reading the amounts and account numbers on paper forms, would otherwise pay a person to type every digit.

**Who would use it:** the team that processes those forms, as the step that turns each scanned digit into a number, with a person checking the cases the model is unsure about.

This is a classification problem with ten classes.

**In an interview:** "I start every model with a problem statement: what is predicted, why it matters and who would use it. Here: a handwritten digit read from its pixels, for an organisation that processes paper forms."

### 6.2 The dataset and the split

**MNIST** is a standard set of 70,000 small greyscale images of handwritten digits, each 28 by 28 pixels and labelled with its digit. `tf.keras.datasets.mnist.load_data()` downloads it (about 11 MB, on the first run only) and returns 60,000 training images and 10,000 test images. Each pixel is a whole number from 0 (black) to 255 (white).

We split the data into three sets before doing anything else:

- the **training set**: the rows the weights learn from;
- the **validation set**: rows the model is scored on during training but never learns from. Every choice we make (how long to train, which model to keep) is made on these;
- the **test set**: rows held back for one final score.

> **The test set is held out now and used once, at the end; every choice before then is made on the validation data.**

MNIST already comes with its test set apart. For validation we keep the last 6,000 training images. They are the same 6,000 rows every day this week, which makes every before-and-after comparison fair, so we always pass them to `fit` as `validation_data` and never use `validation_split`.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(f"Training images {x_train.shape}, test images {x_test.shape}, pixels from {x_train.min()} to {x_train.max()}")

# The test set is held out now and used once, at the end (section 8.2).
x_tr, y_tr = x_train[:54000], y_train[:54000]      # training rows: the weights learn from these
x_val, y_val = x_train[54000:], y_train[54000:]    # validation rows: every choice is made on these
print(f"Training rows {len(x_tr):,}, validation rows {len(x_val):,}, test rows {len(x_test):,} (put aside)")

A look at the data: fifteen training images with their labels, and how many training images there are of each digit.

In [ ]:
sample = slice(0, 15)                                 # Try changing it: slice(15, 30)
show_image_grid(x_tr[sample], [f"label {label}" for label in y_tr[sample]])

counts = pd.Series(y_tr).value_counts().sort_index()
print("Training images per digit:", counts.to_dict())
print(f"Share of 5s among the validation rows: {np.mean(y_val == 5):.3f}")

- 54,000 training rows, 6,000 validation rows and 10,000 test rows. `x_test` and `y_test` are not scored on again until section 8.2.
- The ten digits are roughly balanced in the training rows, from 4,872 images (digit 5) to 6,112 (digit 1). The 5s are 0.091 of the validation rows, about one image in eleven: section 7.3 comes back to that.
- The images are centred, about the same size, and white on a black background. Remember that for the limitations in section 8.3.
- Try changing the slice to `slice(15, 30)` to see fifteen other digits. Put it back.

**In an interview:** "I split before anything else. The test set is put aside and used once at the end, and a fixed validation set is what every choice is made on, so the final test score is an honest estimate."

### 6.3 Preprocessing

Two steps, applied the same way to all three sets:

- **Flattening.** A Dense layer reads one row of numbers per example, so each 28 by 28 grid becomes one row of 784 numbers. In `reshape(-1, 784)`, the `-1` means "as many rows as there are images".
- **Scaling.** Dividing by 255.0 puts every pixel between 0 and 1, the range the Keras beginner tutorial uses. `astype("float32")` first turns the whole numbers into 32-bit decimals, the type the network reads.

Run this cell once. Running it a second time would divide by 255 again; if that happens, rerun the split cell in 6.2 first.

In [ ]:
x_tr = x_tr.reshape(-1, 784).astype("float32") / 255.0
x_val = x_val.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0      # the same steps; nothing is scored on it yet

print(f"Shapes: training {x_tr.shape}, validation {x_val.shape}, test {x_test.shape}")
print(f"Pixels now from {x_tr.min()} to {x_tr.max()}, dtype {x_tr.dtype}")

- Every image is now one row of 784 numbers between 0.0 and 1.0.
- Forgetting the division is a common mistake: the network then reads pixels from 0 to 255 and trains badly.
- Flattening throws away which pixels are neighbours. Section 8.3 comes back to that.

**In an interview:** "My preprocessing flattened each 28 by 28 image into 784 numbers and divided by 255 so every pixel lies between 0 and 1, with the same steps on the training, validation and test data."

### 6.4 The model and a reason for each choice

`digit_model` is a 784-128-10 network: 784 inputs, one hidden layer of 128 units, 10 outputs. Two pieces are new:

- **Softmax output.** Friday named softmax: it turns ten scores into ten probabilities, one per digit, that add up to 1. **argmax** (the position of the largest value) then picks the predicted digit.
- **Sparse categorical cross-entropy.** Section 1.2's cross-entropy for many classes, with the labels given as the integers 0 to 9. Its sibling `categorical_crossentropy` expects each label as ten numbers with a single 1 (**one-hot**); mixing the two up is a common error.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)

digit_model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation="relu"),        # Try changing it: Dense(32, ...)
    tf.keras.layers.Dense(10, activation="softmax"),
])
digit_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
digit_model.summary()

`summary()` shows 101,770 parameters: 784 × 128 + 128 = 100,480 in the hidden layer and 128 × 10 + 10 = 1,290 in the output layer. Each choice has a reason, and your project gives one for each:

| Choice | What | Reason |
|---|---|---|
| Layers | one hidden `Dense(128)` | the simplest MLP: one hidden layer, at a common starting width |
| Hidden activation | `relu` | the bend that keeps the two layers from collapsing into one; cheap to compute |
| Output layer | `Dense(10, activation="softmax")` | ten classes, so ten probabilities that add up to 1 |
| Loss | `sparse_categorical_crossentropy` | cross-entropy for a class, with labels as the integers 0 to 9 |
| Optimizer | `adam` | gradient descent with a step size that adapts for each weight; a sound default |

Try changing `Dense(128` to `Dense(32` and rerun the cell without fitting: the count drops to 25,450 (784 × 32 + 32 = 25,120, plus 32 × 10 + 10 = 330). Put it back to 128 and rerun the cell before 6.5.

**In an interview:** "For each choice I give a reason: one hidden ReLU layer as the simplest network that can bend, a ten-unit softmax output because there are ten classes, sparse categorical cross-entropy because the labels are integers, and Adam as the default optimizer."

### 6.5 Training with both curves on one chart

The settings for `fit`:

- `epochs=10`: ten passes over the 54,000 training rows.
- `batch_size=128`: the **batch size** is the number of rows that go into one update, so one epoch is 422 updates (54,000 divided by 128, rounded up).
- `validation_data=(x_val, y_val)`: after each epoch, `fit` scores the validation rows and records `val_loss` and `val_accuracy` beside `loss` and `accuracy`. The weights never learn from these rows.
- `callbacks=[...]`: a **callback** is an object `fit` calls while it trains. `tf.keras.callbacks.TensorBoard(log_dir="logs/fit")` writes the loss and accuracy of each epoch to the folder `logs/fit`, which section 9 opens.

`verbose=2` prints one line per epoch. The fit takes under a minute on a CPU runtime.

In [ ]:
import shutil
shutil.rmtree("logs/fit", ignore_errors=True)       # an empty log folder, so a rerun is not drawn twice

history = digit_model.fit(
    x_tr, y_tr,
    epochs=10,
    batch_size=128,
    validation_data=(x_val, y_val),
    callbacks=[tf.keras.callbacks.TensorBoard(log_dir="logs/fit")],
    verbose=2,
)

In [ ]:
plot_history(history, "loss", "784-128-10 on MNIST: loss")    # Try changing it: "accuracy"

`evaluate` on the validation rows gives the numbers we write down. The build keeps a **results table** as it goes: one row per trained model, scored on the same validation rows.

In [ ]:
val_loss, val_accuracy = digit_model.evaluate(x_val, y_val, verbose=0)
print(f"Validation loss {val_loss:.4f}, validation accuracy {val_accuracy:.4f}")

results = []                                   # one row per trained model, kept as we go
results.append({"model": "784-128-10", "val_loss": val_loss, "val_accuracy": val_accuracy})

- Both curves fall in the first epochs and stay close together: the network is learning digits in general, and the rows it has never learned from agree.
- Over the last few epochs the validation loss flattens out at about 0.083 while the training loss keeps falling, to about 0.031. The gap that opens there is the first sign of what 6.6 does on purpose.
- The validation accuracy is about 97.6%: roughly 150 of the 6,000 validation digits are wrong.
- Try changing the plot to `plot_history(history, "accuracy", ...)`. It redraws without refitting: both accuracy curves rise and stay close. Put it back to `"loss"`.
- Rerunning the `fit` cell on its own carries on from the trained weights, so a second run is really epochs 11 to 20. To start again, rerun the 6.4 cell first.

**In an interview:** "I train against a fixed validation set and draw the training and validation loss on one chart. Both falling together means the model is learning; validation turning up while training keeps falling means it is overfitting."

### 6.6 Overfitting on purpose

**Overfitting** is a model memorising its training rows, noise included, so that it keeps getting better on them and worse on rows it has not seen. A network with 101,770 weights can memorise a small training set easily. To see it, we train the same 784-128-10 shape on only the first 1,000 training rows, for 50 epochs, against the same 6,000 validation rows. The fit takes under a minute.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
n_rows = 1000                                       # Try changing it: 2000

small_model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
small_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
small_history = small_model.fit(x_tr[:n_rows], y_tr[:n_rows], epochs=50, batch_size=128,
                                validation_data=(x_val, y_val), verbose=0)
plot_history(small_history, "loss", f"784-128-10 on {n_rows:,} training rows, 50 epochs: loss")

val_losses, train_losses = small_history.history["val_loss"], small_history.history["loss"]
best_epoch = int(np.argmin(val_losses)) + 1
print(f"Lowest validation loss {min(val_losses):.3f}, at epoch {best_epoch}; at epoch 50 it is {val_losses[-1]:.3f}")
print(f"Training loss at epoch {best_epoch}: {train_losses[best_epoch - 1]:.3f}; at epoch 50: {train_losses[-1]:.3f}")
print("Validation loss at epochs 10, 20, 30, 40, 50:", [round(val_losses[e - 1], 3) for e in (10, 20, 30, 40, 50)])

- The training loss keeps falling all the way to epoch 50, to about 0.014.
- The validation loss falls until about epoch 24 (about 0.345), then turns and climbs to about 0.365 by epoch 50. The rise is small on the chart, so the last printed line lists it every ten epochs: about 0.38, 0.346, 0.348, 0.356 and 0.365. From the turn on, the network is learning these 1,000 rows instead of digits.
- A chart of the training loss alone would hide this: it looks best exactly when the model is most overfitted. That is why both curves go on one chart.
- **Early stopping** halts training when the validation loss stops improving and can keep the weights from the best epoch. Wednesday uses it in code.
- Try changing `n_rows` to 2000. With twice the rows the network has more to learn before memorising takes over: in our run the validation loss bottoms out lower, at about 0.28 around epoch 21, and still turns up after it, ending at about 0.30. Put it back to 1000.

**In an interview:** "Overfitting shows on the curves as the validation loss turning up while the training loss keeps falling. The usual fixes are stopping at the best validation epoch, which early stopping automates, more training data, or a smaller model."

## 7. Evaluating on the validation data

Accuracy says how often the model is right. Evaluation asks where it is wrong, and how. Everything in this section uses `digit_model` from 6.5 and the validation rows. The test set is still untouched.

### 7.1 The confusion matrix

`predict` returns ten probabilities per row. `argmax(axis=1)` picks, for each row, the column with the largest probability: the predicted digit.

A **confusion matrix** is a table of true classes against predicted classes. scikit-learn's `confusion_matrix(y_true, y_pred)` puts the true digit on the rows and the predicted digit on the columns, and each cell counts the validation images with that pair.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

val_pred = digit_model.predict(x_val, verbose=0).argmax(axis=1)    # the predicted digit for each row
cm = confusion_matrix(y_val, val_pred)                              # Try changing it: (val_pred, y_val)

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False)
ax.set_title("Validation digits: true (rows) against predicted (columns)")
plt.show()

mistakes = cm.copy()
np.fill_diagonal(mistakes, 0)                       # keep only the wrong answers
print(f"Correct: {np.trace(cm):,} of {cm.sum():,}")
for flat in np.argsort(mistakes, axis=None)[::-1][:3]:
    true_digit, predicted_digit = np.unravel_index(flat, cm.shape)
    print(f"true {true_digit} predicted as {predicted_digit}: {mistakes[true_digit, predicted_digit]} times")

- The diagonal, from top left to bottom right, counts the correct answers: 5,854 of 6,000 in this run. Every other cell counts one kind of mistake.
- In this run the largest mistake cells are 5s read as 3 (14 times), then 9s read as 3 and 3s read as 8 (9 times each). 3, 5, 8 and 9 share curves and loops, and a few pixels of stroke separate them.
- Try changing the arguments to `confusion_matrix(val_pred, y_val)`. The chart flips across the diagonal, and the rows become the predictions. Put it back: the true labels go first.

**In an interview:** "A confusion matrix puts the true classes on the rows and the predictions on the columns. The diagonal is the correct answers, and the largest cells off it show which classes the model mixes up."

### 7.2 Precision, recall and F1 for one digit

Ask one yes-or-no question of every image: is it a 5? Then each image falls into one of four boxes, and three of them matter here:

- **true positive (TP):** a 5 the model called 5;
- **false positive (FP):** another digit the model called 5;
- **false negative (FN):** a 5 the model called something else.

From those counts:

- **Precision** = TP / (TP + FP): of the images the model called 5, the share that really are 5s.
- **Recall** = TP / (TP + FN): of the real 5s, the share the model found.
- **F1** combines precision and recall into one number (their harmonic mean, an average pulled towards the smaller of the two), so it is high only when both are.

All three lie between 0 and 1, and higher is better.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

digit = 5                                           # Try changing it: 8
is_five_true = y_val == digit
is_five_pred = val_pred == digit

tp = np.sum(is_five_true & is_five_pred)            # a 5 called 5
fp = np.sum(~is_five_true & is_five_pred)           # another digit called 5
fn = np.sum(is_five_true & ~is_five_pred)           # a 5 called something else
print(f"Digit {digit}: TP {tp}, FP {fp}, FN {fn}")
print(f"precision {precision_score(is_five_true, is_five_pred):.3f}   TP / (TP + FP) = {tp / (tp + fp):.3f}")
print(f"recall    {recall_score(is_five_true, is_five_pred):.3f}   TP / (TP + FN) = {tp / (tp + fn):.3f}")
print(f"F1        {f1_score(is_five_true, is_five_pred):.3f}")

- For digit 5 in this run: 521 true positives, 8 false positives and 28 false negatives. Precision is about 0.985: when the model says 5, it is almost always right. Recall is about 0.949: it misses about one real 5 in twenty, most often reading it as a 3, the top cell of the confusion matrix. F1, about 0.967, sits between the two, nearer the smaller. Precision above recall means this model misses 5s more often than it invents them.
- scikit-learn's numbers match the counts worked by hand, so the definitions above are all there is to it.
- Try changing `digit` to 8 (it also changes 7.3). Compare 8's precision, recall and F1 with 5's and say which digit the model handles better. Put it back to 5.

**In an interview:** "Precision is, of the items the model flagged as the class, the share that were right; recall is, of the real members of the class, the share it found. F1 combines the two and is high only when both are."

### 7.3 When accuracy misleads

A problem is **unbalanced** when one class is much rarer than the others. "Is it a 5?" is unbalanced: about one image in eleven is a 5. Here is a "model" that never looks at a pixel and answers "not 5" every time.

In [ ]:
always_not_five = np.zeros_like(is_five_true)       # answers "not 5" for every image
print(f"Always 'not {digit}':  accuracy {np.mean(always_not_five == is_five_true):.3f}, "
      f"recall {recall_score(is_five_true, always_not_five):.3f}")
print(f"digit_model on 'is it a {digit}?': accuracy {np.mean(is_five_pred == is_five_true):.3f}, "
      f"recall {recall_score(is_five_true, is_five_pred):.3f}")

- The always-"not 5" model scores 0.908 accuracy, about 91%, and finds none of the 5s: recall 0.000.
- On an unbalanced problem, accuracy rewards ignoring the rare class. Recall counts the misses, so it exposes the trick at once. What makes `digit_model` worth having is its recall of about 0.95 for the 5s.

**In an interview:** "On an unbalanced problem accuracy can look high for a useless model: always answering 'not 5' scores about 91% with recall 0. So I report recall, precision and the confusion matrix beside accuracy."

### 7.4 Error analysis

**Error analysis** means looking at the examples the model gets wrong and asking what they share. A shared cause is something the next attempt can aim at. The cell finds every wrong validation row and draws the first fifteen, titled with the true and the predicted digit.

In [ ]:
wrong = np.where(val_pred != y_val)[0]               # positions of the wrong validation rows
print(f"Wrong on {len(wrong)} of {len(y_val):,} validation digits")

shown = wrong[:15]                                   # Try changing it: wrong[15:30]
show_image_grid(x_val[shown], [f"true {y_val[i]}, predicted {val_pred[i]}" for i in shown])

The model is wrong on 146 of the 6,000 validation digits in this run, about 2.4%. Read the grid for what several mistakes share. In our run:

- Most are digits a person reads easily that are written in a less common way: a 7 with a crossbar and a long slanted tail, read as a 2 or a 3; a 1 drawn with a flag and a foot, read as a 2.
- Many are the pairs the confusion matrix already showed: 3s and 8s that differ by a few pixels of loop, a 5 read as a 3.
- Some strokes are much thicker or much thinner than usual (the solid, blotchy 3 read as 8, a thin open 9 read as 1), and some digits are tilted.

Your copy may show a slightly different fifteen, because Colab's library versions differ a little from ours; the kinds of mistake come back the same.

Try changing the slice to `wrong[15:30]` and check whether the same kinds of mistake come back. Put it back.

**In an interview:** "After the metrics I look at the examples the model gets wrong and write down what they share. Here many were digits written in a less common style, 3s, 5s and 8s that differ by a few pixels, and unusually thick or thin strokes, which points at what the next model has to handle."

## 8. One improvement attempt and the test set

### 8.1 One change, measured

One change at a time, so that any difference has one cause. `deeper_model` adds a second hidden layer, `Dense(64, activation="relu")`, after the first: the shape becomes 784-128-64-10. Everything else is as in 6.4 and 6.5: the same seed, 10 epochs, `batch_size=128` and the same 6,000 validation rows. The fit takes under a minute.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)

deeper_model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),         # the one change
    tf.keras.layers.Dense(10, activation="softmax"),
])
deeper_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(f"Parameters: {deeper_model.count_params():,} (784-128-10 has {digit_model.count_params():,})")

deeper_history = deeper_model.fit(x_tr, y_tr, epochs=10, batch_size=128,
                                  validation_data=(x_val, y_val), verbose=2)
plot_history(deeper_history, "loss", "784-128-64-10 on MNIST: loss")

Now its row in the results table, beside the first model's. We compare the validation loss first, because it is the number training lowers, and keep the model with the lower one, chosen here in one line before the test set is touched.

In [ ]:
val_loss, val_accuracy = deeper_model.evaluate(x_val, y_val, verbose=0)
results.append({"model": "784-128-64-10", "val_loss": val_loss, "val_accuracy": val_accuracy})
print(pd.DataFrame(results).round(4))

best = min(results, key=lambda row: row["val_loss"])      # the lower validation loss is kept
kept_model = deeper_model if best["model"] == "784-128-64-10" else digit_model
print(f"Kept model: {best['model']}")

- The deeper model has 109,386 parameters: 100,480 + 8,256 (128 × 64 + 64) + 650 (64 × 10 + 10).
- It did not help. The deeper model's validation loss is about 0.098 against about 0.083, and its validation accuracy about 97.5% against about 97.6%, so the line above keeps `784-128-10`. Its loss chart shows why: its validation loss stopped falling around epoch 8 while its training loss kept going down, so the extra layer mostly let it memorise sooner. A change that did not help is still a result, and it stays in the table. If your copy's table comes out the other way round, the same line keeps the deeper model and the rest of the build runs unchanged.
- On 6,000 validation rows an accuracy gap of 0.001 is six images, so small gaps are read with care.
- This section has no "Try changing" on purpose: a second change at the same time would hide which one did the work. Friday's page goes through the other levers (more epochs, the learning rate, dropout).

**In an interview:** "I changed one thing, a second hidden layer, trained it the same way, and compared both on the same validation rows in a results table. I kept the one with the lower validation loss and said plainly whether the change helped."

### 8.2 The test set, once

This is the only cell that scores `x_test` and `y_test`. It scores the kept model and nothing else.

In [ ]:
test_loss, test_accuracy = kept_model.evaluate(x_test, y_test, verbose=0)   # the one use of the test set
print(f"{best['model']} on the test set: loss {test_loss:.4f}, accuracy {test_accuracy:.4f}")

- The test accuracy is about 97.4%, close to the validation accuracy of about 97.6%. This is the honest estimate of how the model does on digits it has never met.
- It is also this week's reference number: Wednesday's CNN prints its own test accuracy beside it.
- Run it once. If you changed the model after reading this score, the test set would become a second validation set and the score would stop being an honest estimate.

**In an interview:** "The test set is used once, on the model I kept, and gives the honest estimate. If I changed the model after reading it, it would become a second validation set."

### 8.3 Limitations

1. The model reads only digits like MNIST's: centred, about the same size, white on a black background, so a dark digit on white paper or a digit in a corner of the image is outside what it learned.
2. Flattening each image into 784 numbers throws away which pixels are neighbours, which a convolutional network (CNN) keeps; Wednesday builds one on the same digits.

**In an interview** ("Walk me through a model you built"): "I built a digit classifier on MNIST. I wrote the problem statement, held the 10,000 test images out first and kept the last 6,000 training images for validation. I flattened each image to 784 numbers scaled to 0 to 1, and used one hidden ReLU layer of 128 units, a ten-way softmax output, sparse categorical cross-entropy and Adam, with a reason for each. I trained for ten epochs with both loss curves on one chart and reached about 97.6% validation accuracy. I read the confusion matrix, precision, recall and F1 for one digit, and the misclassified images. I tried one change, a second hidden layer, compared both on the same validation rows, kept the better one and scored it on the test set once, about 97.4%. Its limits: it only reads centred digits like MNIST's, and flattening ignores which pixels are neighbours."

## 9. TensorBoard: training curves

**TensorBoard** is the browser dashboard you opened on Thursday of week 6 for its Graphs tab, which showed a model's structure. Its **Scalars** tab draws numbers logged during training, epoch by epoch: how the model learned.

The callback in 6.5 wrote the loss and accuracy of each epoch to `logs/fit`, with the training and the validation numbers in separate subfolders. The two lines below are notebook commands (a line starting with `%` is run by the notebook itself, not by Python): the first loads TensorBoard into the notebook, the second opens it on that folder.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

- Open the Scalars tab and the `epoch_loss` chart. It shows two lines, training and validation: the same curves `plot_history` drew in 6.5.
- The **smoothing** slider averages neighbouring points. Drag it to 0 to see each epoch's value exactly as `history.history` holds it.
- An empty tab usually means the `--logdir` does not match the callback's `log_dir`, or the fit ran without the callback. If the runtime restarted after 6.5, the folder is gone: click this cell and choose Runtime > Run before.
- The Graphs tab of week 6 showed structure; the Scalars tab shows learning. Thursday's page puts two runs side by side here.

**In an interview:** "TensorBoard reads the log folder that Keras's TensorBoard callback writes, and its Scalars tab plots the training and validation loss per epoch. An empty tab usually means the `--logdir` does not match the callback's `log_dir`."

## Recap

| Idea | Type | Good at | Watch out for |
|---|---|---|---|
| Loss (MSE, cross-entropy) | one number for how wrong | MSE for a number; cross-entropy for a class, costly when confident and wrong | read a loss against another loss on the same rows; a Keras loss averages over the batch |
| Gradient descent | training method | lowering the loss one small step at a time: `w - learning_rate * slope` | it only knows the slope where it stands |
| Learning rate | hyperparameter | setting the step size | too small crawls; too large overshoots and diverges |
| Single perceptron | linear classifier | classes one straight line separates | never reaches 1.0 on XOR, however long it trains |
| Multi-layer perceptron | supervised, Keras model | bent boundaries, from a hidden layer with a non-linear activation | without the activation, stacked layers collapse into one line; more units can memorise |
| Backpropagation | how slopes are computed | every weight's slope from one forward and one backward pass | it finds the slopes; the optimizer (SGD, Adam) takes the step |
| Training, validation and test sets | the split | choices on validation rows; one honest test score at the end | never use the test set to choose; fix the validation rows |
| Training and validation curves | diagnosis | both falling: learning; validation turning up: overfitting | a training curve alone hides overfitting |
| Confusion matrix, precision, recall, F1 | evaluation | which classes get mixed up; one class's hits and misses | accuracy misleads on an unbalanced question |
| Error analysis, one change, test once | project method | finding what the mistakes share; a fair before and after | one change at a time, on the same validation rows |
| TensorBoard Scalars | tooling | the same curves from the callback's log folder | an empty tab means the `--logdir` and the `log_dir` differ |

**Quick check**
1. A model gives the true class a probability of 0.9, another gives it 0.1. Which loss do you use for a class, and why is the second so much worse?
2. From the first epoch the training loss bounces and then climbs. What is the likely cause, and what do you change?
3. Why can one neuron not learn XOR, and what fixes it?
4. Explain backpropagation in plain words. In the two-weight network, why is `w1`'s slope three times `w2`'s?
5. Training loss falls while validation loss rises. What is happening, and where would you stop?
6. A model that always answers "not 5" scores about 91% accuracy. What is its recall for 5, and what does that teach you about reporting a classifier?